# 00 · Prepare — cohort, strata, folds, freeze, preprocessing

**Run once, on CPU.** Everything downstream reads what this notebook freezes, and nothing
downstream is meaningful if this is re-run afterwards: each script here refuses to overwrite
its own frozen section, so a second run errors rather than quietly re-rolling the folds.

In order: the pipeline smoke test, the PANORAMA download, content-hash deduplication, the
volume and CNR strata, the patient-level folds, the leave-one-source-out folds, the nnU-Net
dataset, preprocessing, and the matched-budget architecture record.

## The two limits, and how this design stays under them

Kaggle kills a session at 12 hours and refuses to save an output over 20 GB. Both fail
silently — the session simply ends, or the output simply does not save — so both are budgeted
with headroom (11 h, 17 GB) and checked by `enforce_output_budget()` and `check_time()` rather
than hoped for.

The important structural point: **the 20 GB cap is per notebook output, and the worker
notebook is launched once per training run.** Twenty-five runs are twenty-five separate
outputs of ~2 GB each, not one 50 GB pile. Fanning out for concurrency is the same move that
fixes storage.

What goes where:

| | Location | Capped? | Holds |
| --- | --- | --- | --- |
| Output | `/kaggle/working` | **yes, 20 GB** | frozen config, splits, one checkpoint per run, predictions, results |
| Scratch | `/kaggle/temp` | no (wiped) | raw images, `nnUNet_preprocessed`, occluded volumes |

Preprocessed data is deliberately *not* in the output: it is large and exactly regenerable.
For a cohort small enough it is regenerated per session; for the full cohort it is staged as
an attached dataset (last section).

In [ ]:
# === PDAC study bootstrap =============================================================
# Identical in all three notebooks. Paths, environment, repo, disk budget, and the
# cross-session state helpers.
import os, sys, subprocess, shutil, json, tarfile, textwrap, time
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
REPO_URL  = "https://github.com/spraldev/pdac-inductive-bias.git"

# --- the two hard limits, in one place -------------------------------------------------
# Kaggle kills a session at 12 h and refuses to save an output larger than 20 GB. Both are
# silent failures if you meet them by accident, so both are budgeted with headroom and
# checked rather than hoped for.
SESSION_HOURS   = float(os.environ.get("PDAC_SESSION_HOURS", 11.0))   # of a 12 h cap
OUTPUT_LIMIT_GB = float(os.environ.get("PDAC_OUTPUT_LIMIT_GB", 17.0)) # of a 20 GB cap
SESSION_T0 = time.time()

# --- paths ------------------------------------------------------------------------------
# WORK persists as the notebook's saved output and is what the 20 GB cap applies to: only
# small, precious, or genuinely needed-downstream things go there. SCRATCH is much larger and
# is wiped with the session, so everything regenerable lives there — raw images, nnU-Net
# preprocessed data, occluded volumes.
if ON_KAGGLE:
    WORK    = Path("/kaggle/working")
    SCRATCH = Path("/kaggle/temp/pdac"); SCRATCH.mkdir(parents=True, exist_ok=True)
    REPO    = WORK / "pdac-research"
else:
    WORK    = Path(os.environ.get("PDAC_WORK", Path.cwd() / "pdac_work"))
    SCRATCH = Path(os.environ.get("PDAC_SCRATCH", WORK / "scratch"))
    REPO    = Path(os.environ.get("PDAC_REPO", Path.cwd()))
    WORK.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path(os.environ.get("PDAC_DATA", SCRATCH / "data"))
RESULTS   = WORK / "results";  RESULTS.mkdir(parents=True, exist_ok=True)
STATE     = WORK / "state";    STATE.mkdir(parents=True, exist_ok=True)
PREDS     = WORK / "preds";    PREDS.mkdir(parents=True, exist_ok=True)

# --- repo ---------------------------------------------------------------------------------
def _find_attached(*required):
    """First attached input containing all of the given relative paths."""
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for p in sorted(root.glob("*")):
        for cand in [p] + sorted(x for x in p.glob("*") if x.is_dir()):
            if all((cand / r).exists() for r in required):
                return cand
    return None

if ON_KAGGLE and not (REPO / "config" / "analysis_config.yaml").exists():
    src = _find_attached("config/analysis_config.yaml")
    if src is not None:
        print(f"Using repo attached as a dataset: {src}")
        shutil.copytree(src, REPO, dirs_exist_ok=True)
    else:
        print("Cloning the repo (Internet must be on in notebook settings) ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "scripts" / "analysis"))
sys.path.insert(0, str(REPO / "scripts" / "training"))
os.environ["PDAC_REPO"] = str(REPO)

# These notebooks are generated FROM the repo but, on Kaggle, run AGAINST a clone of it — so
# an uncommitted or unpushed change is invisible here no matter how current the notebook is.
# When the clone predates the notebook the symptom lands far from the cause: an old test
# asserting old counts, a script missing a flag this notebook passes. Checking the contract
# up front turns that into one clear message.
_REQUIRED = ["config/analysis_config.yaml", "scripts/training/tasks.py",
             "scripts/analysis/build_per_case_table.py", "scripts/analysis/make_figures.py",
             "src/trainers/budget_trainers.py", "scripts/kaggle/pack_for_kaggle.py"]
_missing = [r for r in _REQUIRED if not (REPO / r).exists()]
if _missing:
    raise RuntimeError(
        "The repo this notebook is running against is older than the notebook itself.\n"
        f"  missing: {_missing}\n"
        f"  repo:    {REPO}\n"
        "These notebooks clone " + REPO_URL + ", so local commits only reach Kaggle "
        "once they are PUSHED. "
        "Either push, or upload the repo as a Kaggle Dataset and attach it "
        "(the bootstrap prefers an attached input containing config/analysis_config.yaml).")

# --- nnU-Net environment --------------------------------------------------------------------
# raw and preprocessed are regenerable and enormous -> SCRATCH.
# results holds checkpoints, which are neither -> WORK, under the budget guard below.
os.environ["nnUNet_raw"]          = str(SCRATCH / "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = str(SCRATCH / "nnUNet_preprocessed")
os.environ["nnUNet_results"]      = str(WORK / "nnUNet_results")
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    Path(os.environ[k]).mkdir(parents=True, exist_ok=True)

# --- shell / install helpers -------------------------------------------------------------------
def sh(cmd, cwd=None, check=True):
    """Run a shell command from the repo root, streaming output into the notebook."""
    cwd = str(cwd or REPO)
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

def pip_install(pkgs, quiet=True):
    sh(f"{sys.executable} -m pip install {'-q ' if quiet else ''}--no-warn-script-location {pkgs}")

def gpu_info():
    try:
        import torch
    except ImportError:
        print("torch not installed yet"); return None
    if not torch.cuda.is_available():
        print("No CUDA device. Turn on a GPU accelerator in notebook settings."); return None
    name = torch.cuda.get_device_name(0)
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  ({gb:.1f} GB)")
    return {"name": name, "vram_gb": round(gb, 1)}

# --- the 20 GB guard ------------------------------------------------------------------------------
def dir_gb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e9

def disk_report(detail=True):
    """What the session is holding, split by which limit it counts against."""
    out = dir_gb(WORK)
    free_scratch = shutil.disk_usage(SCRATCH).free / 1e9
    print(f"OUTPUT (counts against the {OUTPUT_LIMIT_GB:.0f}/20 GB cap): {out:.2f} GB")
    if detail:
        for sub in sorted(p for p in WORK.iterdir() if p.is_dir()):
            g = dir_gb(sub)
            if g > 0.01:
                print(f"    {g:7.2f} GB  {sub.name}/")
    print(f"SCRATCH (wiped with the session, not capped): {dir_gb(SCRATCH):.2f} GB used, "
          f"{free_scratch:.0f} GB free")
    return out

def enforce_output_budget(limit_gb=None, where=""):
    """Get the output back under budget, and only then complain if it cannot be done.

    Raising alone would not help: Kaggle refuses the save regardless of what the notebook
    thinks, so an over-budget session loses its GPU hours either way. This frees space in
    increasing order of regret and re-measures after each step, so the common case (a
    checkpoint the study never evaluates) is handled silently and only a genuine overrun
    reaches the user.
    """
    limit = OUTPUT_LIMIT_GB if limit_gb is None else limit_gb
    used = dir_gb(WORK)
    if used <= limit:
        print(f"output {used:.2f} / {limit:.0f} GB{' at ' + where if where else ''}  OK")
        return used

    print(f"output {used:.2f} GB is over the {limit:.0f} GB budget — freeing space")

    # 1. Checkpoints this study never reads. Zero regret.
    prune_checkpoints()
    used = dir_gb(WORK)

    # 2. Softmax dumps and validation scratch. Nothing here reads them either; they only
    #    appear if a training command was run with --npz, which this repo no longer does.
    if used > limit:
        for pattern in ("*.npz", "*.pkl"):
            for f in Path(os.environ["nnUNet_results"]).rglob(pattern):
                print(f"  removed {f.name}"); f.unlink()
        used = dir_gb(WORK)

    # 3. Resume checkpoints for runs that finished. Costs the ability to resume a run that
    #    has nothing left to resume.
    if used > limit:
        prune_checkpoints(keep_latest_if_unfinished=False)
        used = dir_gb(WORK)

    if used > limit:
        disk_report()
        raise RuntimeError(
            f"Output is still {used:.1f} GB after pruning, over the {limit:.0f} GB budget"
            f"{' at ' + where if where else ''}. Kaggle will refuse to save this session. "
            "Drop this task's checkpoint (DROP_CHECKPOINT = True) if its predictions are "
            "already written — every analysis except the receptive-field measurement reads "
            "predictions, not weights.")
    print(f"output now {used:.2f} / {limit:.0f} GB  OK")
    return used

def time_left_h():
    return SESSION_HOURS - (time.time() - SESSION_T0) / 3600.0

def check_time(where=""):
    left = time_left_h()
    print(f"{left:.2f} h left of the {SESSION_HOURS:.1f} h session budget"
          f"{' at ' + where if where else ''}")
    return left

def prune_checkpoints(root=None, keep_latest_if_unfinished=True):
    """Keep exactly what the study needs from each run's directory.

    nnU-Net writes checkpoint_best, checkpoint_latest and checkpoint_final. Evaluation in this
    study is on checkpoint_final only (the pre-registered schedule has no early stopping), so
    best is always removable, and latest is removable the moment final exists. Left alone,
    three checkpoints per run is three times the storage for no gain.
    """
    root = Path(root or os.environ["nnUNet_results"])
    freed = 0.0
    for fold_dir in sorted(p for p in root.rglob("fold_*") if p.is_dir()):
        final = fold_dir / "checkpoint_final.pth"
        drop = [fold_dir / "checkpoint_best.pth"]
        if final.exists() or not keep_latest_if_unfinished:
            drop.append(fold_dir / "checkpoint_latest.pth")
        for f in drop:
            if f.exists():
                freed += f.stat().st_size / 1e9
                f.unlink()
                print(f"  removed {f.relative_to(root)}")
    if freed:
        print(f"freed {freed:.2f} GB")
    return freed

# --- cross-session state --------------------------------------------------------------------------
def save_state(*rel_paths):
    """Copy repo-relative paths into WORK/state so they survive as notebook output."""
    for rel in rel_paths:
        src = REPO / rel
        if not src.exists():
            print(f"  (skip, absent) {rel}"); continue
        dst = STATE / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        (shutil.copytree if src.is_dir() else shutil.copy2)(
            src, dst, **({"dirs_exist_ok": True} if src.is_dir() else {}))
        print(f"  saved {rel}")

def restore_state(*rel_paths, required=True):
    """Restore from WORK/state or from any attached dataset holding a state/ directory."""
    sources = [STATE]
    if Path("/kaggle/input").exists():
        sources += sorted(Path("/kaggle/input").rglob("state"))
    missing = []
    for rel in rel_paths:
        for base in sources:
            src = base / rel
            if src.exists():
                dst = REPO / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                (shutil.copytree if src.is_dir() else shutil.copy2)(
                    src, dst, **({"dirs_exist_ok": True} if src.is_dir() else {}))
                print(f"  restored {rel}  <- {base}")
                break
        else:
            missing.append(rel)
    if missing:
        msg = ("Missing state: " + ", ".join(missing) + "\n  Run the preparation notebook, "
               "then attach its output here (Add Input -> Your Work).")
        if required:
            raise FileNotFoundError(msg)
        print("  " + msg)
    return not missing

def restore_chunks(dest, manifest_name="pack_manifest.json"):
    """Unpack a chunked dataset produced by scripts/kaggle/pack_for_kaggle.py.

    Large reusable data (the preprocessed cohort) cannot travel as notebook output — that is
    what the 20 GB cap forbids — so it travels as an attached dataset in size-bounded parts.
    This finds the manifest in any attached input and extracts every part into dest.
    """
    root = Path("/kaggle/input")
    if not root.exists():
        return False
    for man_path in sorted(root.rglob(manifest_name)):
        man = json.loads(man_path.read_text())
        parts = man.get("parts", [])
        print(f"Found a {len(parts)}-part pack at {man_path.parent} "
              f"({man.get('uncompressed_bytes', 0)/1e9:.1f} GB)")
        dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
        for entry in parts:
            src = man_path.parent / entry["name"]
            if not src.exists():
                print(f"  MISSING {entry['name']} — attach every part, not just some"); continue
            with tarfile.open(src) as tar:
                tar.extractall(dest)
            print(f"  extracted {entry['name']} ({entry['n_files']} files)")
        return True
    return False

print(f"ON_KAGGLE={ON_KAGGLE}\nREPO={REPO}\nWORK={WORK}\nSCRATCH={SCRATCH}\n"
      f"DATA_ROOT={DATA_ROOT}\nbudgets: {SESSION_HOURS} h session, {OUTPUT_LIMIT_GB} GB output")

In [ ]:
# Analysis environment. Kaggle already ships numpy/pandas/scipy/matplotlib; these are the rest.
pip_install("SimpleITK nibabel openpyxl pyyaml statsmodels zenodo-get "
            "'surface-distance @ git+https://github.com/google-deepmind/surface-distance.git'")
import importlib
for m in ("SimpleITK", "surface_distance", "statsmodels", "yaml", "pandas", "scipy"):
    importlib.import_module(m)
print("analysis environment OK")

In [ ]:
# nnU-Net from master: the PrimusV2 trainers are not guaranteed to be in the PyPI release.
# The commit is pinned into the frozen config the first time this runs, so every later session
# and every collaborator gets the same one.
import yaml
frozen_path = REPO / "config" / "frozen_thresholds.yaml"
frozen = (yaml.safe_load(open(frozen_path)) or {}) if frozen_path.exists() else {}
pinned = (frozen.get("architecture") or {}).get("nnunet_commit")
pinned = pinned.split("@")[-1].strip() if pinned and "@" in str(pinned) else None

spec = "git+https://github.com/MIC-DKFZ/nnUNet.git" + (f"@{pinned}" if pinned else "")
print(f"Installing nnunetv2 from {spec}")
pip_install(f"'nnunetv2 @ {spec}'")
import nnunetv2
sh("pip freeze | grep -i nnunet")

In [ ]:
# --- the pipeline smoke test ---------------------------------------------------------
# 123 checks over every data, analysis, and training script, against synthetic data with a designed
# answer, so it asserts on the statistical outcome rather than on exit codes. If this is
# green the analysis works, and everything after this is data and compute.
sh(f"{sys.executable} tests/run_smoke_test.py")
check_time("after smoke test")

In [ ]:
# --- configuration --------------------------------------------------------------------
SUBSET = True     # batch 1 only. Set False only where ~400 GB of disk actually exists.
BATCHES = ["batch_1"] if SUBSET else ["batch_1", "batch_2", "batch_3", "batch_4"]

# If PANORAMA is already attached as a dataset, use it and skip the download entirely.
# That is the intended path once you have the data once: uploading it as a dataset is not
# subject to the 20 GB output cap.
attached = _find_attached("panorama/panorama_labels")
if attached:
    print(f"Using attached PANORAMA at {attached}")
    DATA_ROOT = attached
else:
    print(f"Will download: {BATCHES}")
print(f"free scratch: {shutil.disk_usage(SCRATCH).free/1e9:.0f} GB")

In [ ]:
# --- download ---------------------------------------------------------------------------
# zenodo_get resolves each record, downloads every file, and verifies md5 checksums.
if not attached:
    (DATA_ROOT / "panorama" / "images").mkdir(parents=True, exist_ok=True)
    zips = DATA_ROOT / "panorama" / "zips"
    RECORDS = {"batch_1": 13715870, "batch_2": 13742336, "batch_3": 11034011, "batch_4": 10999754}
    for b in BATCHES:
        d = zips / b
        if (d / ".done").exists():
            print(f"{b} already downloaded"); continue
        d.mkdir(parents=True, exist_ok=True)
        sh(f"zenodo_get -o {d} {RECORDS[b]}")
        (d / ".done").touch()
        check_time(f"after {b}")
    for b in BATCHES:
        for z in sorted((zips / b).glob("*.zip")):
            sh(f"unzip -n -q {z} -d {DATA_ROOT / 'panorama' / 'images'}")
    # The zips are dead weight once extracted, and scratch is finite too.
    shutil.rmtree(zips, ignore_errors=True)

    labels = DATA_ROOT / "panorama" / "panorama_labels"
    if not (labels / ".git").exists():
        sh("git lfs install || true", cwd=WORK, check=False)
        sh(f"git clone https://github.com/DIAGNijmegen/panorama_labels {labels}", cwd=WORK)
    commit = subprocess.run(["git", "-C", str(labels), "rev-parse", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
    (DATA_ROOT / "panorama" / "labels_commit.txt").write_text(commit + chr(10))
    print("labels commit:", commit)

print(f"{len(list((DATA_ROOT / 'panorama' / 'images').rglob('*')))} files under images/")
disk_report()

# Everything below writes to scratch, which is wiped when this session ends. A session that
# runs out of time mid-preprocessing therefore loses the download too — there is no partial
# credit here, so the check is up front rather than after the fact.
left = check_time("after download")
if left < 3.0:
    print("*** Less than 3 h left and preprocessing has not started. Scratch does not survive "
          "the session, so finishing this notebook in a later one is not possible: it would "
          "re-download from scratch.")
    print("    Upload the data once as a Kaggle Dataset from a machine with disk, attach it, "
          "and re-run — the download is then skipped entirely and this notebook fits easily. "
          "That is also the only route that works for the full ~190 GB cohort. ***")

In [ ]:
# --- cohort table + content-hash deduplication --------------------------------------------
# Duplicates are found by image content, not case ID: MSD Task07 and NIH Pancreas-CT are
# redistributed inside PANORAMA under different IDs, and ID-based dedup would miss them.
sh(f"{sys.executable} scripts/data/deduplicate.py --data-root {DATA_ROOT} --out splits/cohort.csv")

import pandas as pd, yaml, datetime
cohort = pd.read_csv(REPO / "splits" / "cohort.csv")
dupes  = pd.read_csv(REPO / "splits" / "duplicates.csv")
print(f"{len(cohort)} scans retained, {len(dupes)} exact duplicates removed, "
      f"{cohort['has_manual_lesion'].sum()} with a manual lesion delineation")
if "source" in cohort.columns:
    print(cohort.groupby("source")["case_id"].nunique().to_string())
else:
    print("NO 'source' COLUMN — inspect the clinical_information.xlsx column names printed "
          "above. The source axis, the splits, and leave-one-source-out all depend on it.")

In [ ]:
# Record what this cohort came from, so a subset can never be mistaken for the study cohort.
prov = {
    "built": datetime.date.today().isoformat(),
    "data_root": str(DATA_ROOT),
    "batches": "attached-dataset" if attached else BATCHES,
    "subset_of_full_panorama": bool(SUBSET and not attached),
    "labels_commit": ((DATA_ROOT / "panorama" / "labels_commit.txt").read_text().strip()
                      if (DATA_ROOT / "panorama" / "labels_commit.txt").exists() else None),
    "n_scans_retained": int(len(cohort)),
    "n_duplicates_removed": int(len(dupes)),
    "n_manual_lesion_cases": int(cohort["has_manual_lesion"].sum()),
}
(REPO / "splits" / "cohort_provenance.yaml").write_text(yaml.safe_dump(prov, sort_keys=False))
print(yaml.safe_dump(prov, sort_keys=False))
if prov["subset_of_full_panorama"]:
    print("*** SUBSET cohort. Nothing computed from it is a study result. ***")

In [ ]:
# --- strata: volume, diameter, CNR at 5/10/15 mm in one pass --------------------------------
# One pass over the images gives the primary ring and both sensitivity rings, so the
# pre-specified 5 mm / 15 mm analysis costs no second read.
sh(f"{sys.executable} scripts/analysis/compute_strata.py --data-root {DATA_ROOT} "
   f"--cohort splits/cohort.csv --out splits/strata.csv")

frozen = yaml.safe_load(open(REPO / "config" / "frozen_thresholds.yaml"))
print(yaml.safe_dump(frozen["strata"], sort_keys=False))
if frozen["strata"]["contrast_axis_exploratory"]:
    print("Pre-registered rule fired: CNR ranks are unstable across ring widths, so the "
          "contrast axis is reported as EXPLORATORY. Pre-specified — not a retreat, and not "
          "revisited.")
check_time("after strata")

In [ ]:
# --- folds: patient-level CV, then leave-one-source-out -------------------------------------
# If the raw source values need remapping to the study's canonical groups, set SOURCE_MAPPING;
# every raw value must be covered or make_splits.py fails loudly rather than guessing, and
# whichever grouping is used is frozen into config/frozen_thresholds.yaml.
SOURCE_MAPPING = None   # e.g. {"RUMC": "Radboud", "UMCG": "UMCG", "MSD": "MSKCC", "NIH": "NIH"}
extra = ""
if SOURCE_MAPPING:
    (REPO / "splits" / "source_mapping.json").write_text(json.dumps(SOURCE_MAPPING, indent=1))
    extra = "--source-mapping splits/source_mapping.json"
sh(f"{sys.executable} scripts/data/make_splits.py --cohort splits/cohort.csv "
   f"--strata splits/strata.csv {extra}")

# Deterministic, no seed: each source held out in turn. --min-cases enforces the
# pre-registration's escalation rule rather than emitting a meaningless two-case LOSO fold.
sh(f"{sys.executable} scripts/data/make_loso_splits.py --exclude-source NIH --min-cases 25 "
   f"--emit-combined splits/splits_with_loso.json", check=False)

In [ ]:
splits = json.load(open(REPO / "splits" / "splits_final.json"))
fa = pd.read_csv(REPO / "splits" / "fold_assignment.csv")
print(f"{len(splits)} CV folds over {len(fa)} cases / {fa['patient_id'].nunique()} patients")
print(pd.crosstab(fa["fold"], fa["source"]).to_string() if "source" in fa.columns
      else fa["fold"].value_counts().sort_index().to_string())
loso_csv = REPO / "splits" / "loso_folds.csv"
if loso_csv.exists():
    print("\nLeave-one-source-out folds:")
    print(pd.read_csv(loso_csv).to_string(index=False))
else:
    print("\nNo LOSO folds: the cell above escalated. Decide (merge, drop, or lower the bar), "
          "record the decision, and re-run that cell with it applied.")

In [ ]:
# --- nnU-Net dataset and preprocessing ------------------------------------------------------
# Only manual-lesion cases enter Dataset501; the model-generated delineations are staged
# separately and never enter evaluation. Both live in SCRATCH: large and exactly regenerable.
sh(f"{sys.executable} scripts/data/convert_to_nnunet.py --data-root {DATA_ROOT} "
   f"--cohort splits/cohort.csv")

# Preset pairing. This is the study's only control, so it is measured, not chosen by taste:
# both arms must fit one VRAM ceiling. ResEnc VRAM per the upstream presets doc is M ~9-11 GB,
# L ~24 GB, XL ~40 GB. This notebook has no GPU, so the pairing is provisional here and the
# numbers are measured in the first worker session.
PROVISIONAL_VRAM_GB = 16      # the card the workers will use
if PROVISIONAL_VRAM_GB >= 40:
    CNN_PLANS, PRIMUS_TRAINER = "nnUNetResEncUNetXLPlans", "nnUNet_PrimusV2L_Trainer"
elif PROVISIONAL_VRAM_GB >= 22:
    CNN_PLANS, PRIMUS_TRAINER = "nnUNetResEncUNetLPlans", "nnUNet_PrimusV2M_Trainer"
else:
    CNN_PLANS, PRIMUS_TRAINER = "nnUNetResEncUNetMPlans", "nnUNet_PrimusV2S_Trainer"
PLANNER = "nnUNetPlannerResEnc" + CNN_PLANS.replace("nnUNetResEncUNet", "").replace("Plans", "")
print(f"CNN {CNN_PLANS} (planner {PLANNER}) | transformer {PRIMUS_TRAINER}")

sh(f"nnUNetv2_plan_and_preprocess -d 501 -pl {PLANNER} --verify_dataset_integrity")
check_time("after preprocessing")

In [ ]:
# --- install the frozen splits into the preprocessed dataset ---------------------------------
# Both arms must train on identical folds, so the frozen file is copied in rather than letting
# nnU-Net generate its own. With LOSO folds present the combined file goes in: it keeps the
# five CV folds byte-identical at indices 0-4 and appends LOSO at 5, 6, ...
ds_dir = Path(os.environ["nnUNet_preprocessed"]) / "Dataset501_PDAC"
src = REPO / "splits" / ("splits_with_loso.json"
                         if (REPO / "splits" / "splits_with_loso.json").exists()
                         else "splits_final.json")
shutil.copy(src, ds_dir / "splits_final.json")
print(f"installed {src.name}: {len(json.load(open(ds_dir / 'splits_final.json')))} folds")

plans = json.load(open(ds_dir / f"{CNN_PLANS}.json"))
patch_xyz = plans["configurations"]["3d_fullres"]["patch_size"]
grid = [p // 8 for p in patch_xyz]
print(f"patch {patch_xyz}, divides by the 8x8x8 Primus tokenizer stride: "
      f"{all(p % 8 == 0 for p in patch_xyz)}, token grid {grid} = {grid[0]*grid[1]*grid[2]}")

In [ ]:
# --- how the preprocessed data reaches the workers --------------------------------------------
# This is the decision the 20 GB cap actually forces. Preprocessed data is far too large to be
# notebook output for a real cohort, so there are exactly two honest options, and which one
# applies is a measurement, not a preference.
pre_gb = dir_gb(ds_dir)
print(f"preprocessed Dataset501_PDAC: {pre_gb:.1f} GB")
print(f"raw (SCRATCH): {dir_gb(os.environ['nnUNet_raw']):.1f} GB")

if pre_gb <= OUTPUT_LIMIT_GB - dir_gb(WORK) - 1:
    print("\nSmall enough to travel as this notebook's output. Packing it into parts so the "
          "worker notebook can attach it and skip preprocessing entirely.")
    sh(f"{sys.executable} scripts/kaggle/pack_for_kaggle.py --src {ds_dir} "
       f"--out {WORK / 'preprocessed_pack'} --slug pdac-preprocessed "
       f"--title 'PDAC nnU-Net preprocessed (Dataset501)' --chunk-gb 4")
else:
    print(f"\n{pre_gb:.0f} GB does NOT fit in a notebook output, and no amount of chunking "
          "changes that — the cap is on the whole output.")
    print("Two options, both fine, neither of them 'squeeze it in':")
    print("  A. Let each worker regenerate it. Every worker session then spends its first "
          "hour or two preprocessing before it trains. Correct, just wasteful.")
    print("  B. Stage it once as a dataset, which is NOT subject to the 20 GB cap. On a "
          "machine with the data and the disk:")
    print("       python scripts/kaggle/pack_for_kaggle.py --src <nnUNet_preprocessed> \\")
    print("           --out /tmp/pack --slug pdac-preprocessed --chunk-gb 15")
    print("       cd /tmp/pack && kaggle datasets create -d -r skip")
    print("     Then attach it to the worker notebook; restore_chunks() unpacks it and "
          "preprocessing is skipped.")
disk_report()

In [ ]:
# --- freeze and hand off ------------------------------------------------------------------------
save_state("splits", "config/frozen_thresholds.yaml", "preregistration/DEVIATIONS.md")
enforce_output_budget(where="end of preparation")

# The full task list the worker notebook is launched against, so "which runs are left" is a
# set difference rather than a memory exercise.
sh(f"{sys.executable} scripts/training/tasks.py")

## Before any training: tag `prereg-v1`, on your own machine

Kaggle has no push credentials, and the tag is the whole point of the pre-registration. Copy
`splits/` and `config/frozen_thresholds.yaml` out of this notebook's output, then:

```bash
git add config/frozen_thresholds.yaml splits/
git commit -m "Freeze cohort strata, folds, and source grouping"
git tag -a prereg-v1 -m "Pre-registration frozen before any training run"
git push origin main --tags
bash scripts/check_prereg_tag.sh          # must print PASS
```

That check fails if the tag moves, if history is rewritten under it, or if any frozen file
differs between the tag and HEAD. Run it before every analysis session.

## Then: **Save Version → Save & Run All**

The output carries the frozen config, the splits, and (when it fits) the packed preprocessed
data. Attach this session to every worker.

Next: **01_run**, launched once per task — concurrently.